<a href="https://colab.research.google.com/github/tharak-bairneni/sql-pandas-employee-analysis/blob/main/EmployeeAnalysis_TharakBairneni_2601916.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Task 1: Create Database

In [1]:
import sqlite3
import pandas as pd
import random

# Connect to SQLite database
conn = sqlite3.connect("employees.db")
cursor = conn.cursor()

# Create table
cursor.execute("""
CREATE TABLE IF NOT EXISTS employees (
    emp_id INTEGER PRIMARY KEY,
    name TEXT,
    department TEXT,
    salary REAL,
    years_experience INTEGER,
    performance_score REAL
)
""")

departments = ["Engineering", "Sales", "Marketing", "HR", "Finance"]

# Generate synthetic employee data
employees_data = []

for i in range(1, 41):
    name = f"Employee_{i}"
    department = random.choice(departments)
    salary = random.randint(50000, 150000)
    years_experience = random.randint(1, 15)
    performance_score = round(random.uniform(1.0, 5.0), 2)

    employees_data.append((i, name, department, salary, years_experience, performance_score))

# Insert records
cursor.executemany("""
INSERT INTO employees
(emp_id, name, department, salary, years_experience, performance_score)
VALUES (?, ?, ?, ?, ?, ?)
""", employees_data)

conn.commit()

print("✓ Database and employee table created with 40 records")

✓ Database and employee table created with 40 records


Task 2: SQL Queries

Query 1

Employees with high performance and experience.

In [2]:
query1 = """
SELECT name, department, salary, performance_score
FROM employees
WHERE performance_score >= 4.0
AND years_experience >= 3
ORDER BY performance_score DESC
LIMIT 15
"""

df_query1 = pd.read_sql(query1, conn)
df_query1

,name,department,salary,performance_score
0,Employee_7,Finance,56204.0,4.97
1,Employee_11,Engineering,115985.0,4.86
2,Employee_37,Sales,52213.0,4.82
3,Employee_21,Marketing,84638.0,4.77
4,Employee_10,Engineering,121969.0,4.75
5,Employee_29,Marketing,103653.0,4.70
6,Employee_20,Engineering,127896.0,4.68
7,Employee_33,Marketing,60979.0,4.58
8,Employee_22,Sales,92197.0,4.24
9,Employee_18,Finance,115427.0,4.22


Query 2

Salary between range in Engineering or Sales.

In [3]:
query2 = """
SELECT *
FROM employees
WHERE salary BETWEEN 70000 AND 110000
AND department IN ('Engineering', 'Sales')
ORDER BY department, salary DESC
"""

df_query2 = pd.read_sql(query2, conn)
df_query2

,emp_id,name,department,salary,years_experience,performance_score
0,17,Employee_17,Engineering,78742.0,4,3.62
1,1,Employee_1,Engineering,76945.0,6,3.24
2,8,Employee_8,Engineering,71195.0,12,2.89
3,35,Employee_35,Engineering,70720.0,14,3.62
4,30,Employee_30,Sales,106130.0,3,2.65
5,22,Employee_22,Sales,92197.0,8,4.24


Query 3

Department statistics.

In [4]:
query3 = """
SELECT department,
COUNT(*) AS employee_count,
AVG(salary) AS avg_salary
FROM employees
GROUP BY department
ORDER BY avg_salary DESC
"""

df_query3 = pd.read_sql(query3, conn)
df_query3

,department,employee_count,avg_salary
0,Engineering,12,110029.250000
1,HR,4,96830.750000
2,Sales,7,93582.714286
3,Marketing,8,92826.875000
4,Finance,9,89150.888889


Task 3: Pandas Analysis

Average Performance by Department

In [5]:
df = pd.read_sql("SELECT * FROM employees", conn)

avg_performance = df.groupby("department")["performance_score"].mean()

avg_performance

,performance_score
department,
Engineering,3.500833
Finance,3.434444
HR,3.560000
Marketing,3.357500
Sales,2.978571


Replicate Query 1 Using Pandas Only

In [6]:
pandas_query1 = (
    df.loc[(df["performance_score"] >= 4.0) & (df["years_experience"] >= 3),
           ["name", "department", "salary", "performance_score"]]
    .sort_values(by="performance_score", ascending=False)
    .head(15)
)

pandas_query1

,name,department,salary,performance_score
6,Employee_7,Finance,56204.0,4.97
10,Employee_11,Engineering,115985.0,4.86
36,Employee_37,Sales,52213.0,4.82
20,Employee_21,Marketing,84638.0,4.77
9,Employee_10,Engineering,121969.0,4.75
28,Employee_29,Marketing,103653.0,4.70
19,Employee_20,Engineering,127896.0,4.68
32,Employee_33,Marketing,60979.0,4.58
21,Employee_22,Sales,92197.0,4.24
17,Employee_18,Finance,115427.0,4.22


SQL vs Pandas Comparison

SQL and Pandas are both powerful tools for data analysis but are used in different contexts. SQL is primarily used for querying and managing data stored in relational databases, while Pandas is a Python library used for data manipulation and analysis in memory.

One key difference is syntax. In SQL, conditions use operators like =, AND, and ORDER BY. In Pandas, similar operations use ==, &, and .sort_values(). SQL queries are declarative, meaning you specify what result you want, while Pandas operations are more procedural and Python-based.

SQL is best used when working directly with databases, especially when data is very large and stored on servers. It efficiently retrieves only the necessary records without loading the entire dataset into memory.

Pandas is ideal for detailed data analysis, transformations, statistical calculations, and visualization preparation once data has been loaded into Python.

For datasets that are too large to fit into memory, SQL is generally the better tool because the database engine processes the data on disk and returns only the required results. Pandas requires data to fit into memory, making it more suitable for moderate-sized datasets.

Close Database Connection

In [7]:
conn.close()